# AcademicComback, Part 1 — The Transcription Pipeline**How this notebook works.** You are going to rebuild your own lecture transcriber from scratch, in Python, one step at a time. Every cell runs and shows you something real. Then, once the logic is in your head, I show you the actual JavaScript from your site sitting right next to the Python you just wrote.Why Python when your site is JavaScript? Because the *hard part* of this project isn't the language — it's the pipeline: how you take a 45-minute recording and turn it into a document. That pipeline is identical in any language. Learn it in Python where you can run one line and immediately see the result, and the JavaScript version stops being mysterious.**A rule for this notebook: don't just run the cells.** Read the cell, guess what it will print, *then* run it. When your guess is wrong, that's the moment you actually learn something. Running everything top-to-bottom without thinking will teach you nothing, and you'll know it.---### What you'll be able to do when you finish- Explain, out loud and without notes, every single thing that happens between "user picks a file" and "user downloads a .docx"- Split an audio file at natural pauses using ffmpeg from the command line- Call the Groq Whisper API by hand, and explain what every parameter does- Explain why your API key is not in your JavaScript, and what would happen if it were- Build a .docx with code- Rebuild this whole tool yourself without asking an AI for anything

---## Part 1 — The mapBefore any code: what actually happens when a student uses your site?```      STUDENT'S BROWSER                          THE INTERNET      ─────────────────────────────────          ──────────────────────  1.  picks lecture.m4a  (45 min, 40 MB)                │  2.  ffmpeg.wasm looks at the audio and      finds every natural pause                │  3.  cuts it into ~13-minute pieces at      those pauses, squashing each one      down to a tiny mono mp3                │                ├── piece 1 ──────────────►  your Netlify Function                ├── piece 2 ──────────────►  (/api/transcribe)                ├── piece 3 ──────────────►         │                └── piece 4 ──────────────►         │  adds your secret                                                    │  API key, forwards on                                                    ▼                                             Groq's Whisper API                                                    │                ◄─────── text comes back ───────────┘                │  4.  glues the pieces back together in order                │  5.  builds a .docx in the browser                │  6.  browser downloads it```**Five things worth noticing right now**, because each one is a deliberate decision you made, and you should be able to defend all of them:1. **The heavy work happens on the student's device, not your server.** Splitting a 40 MB file takes real processing. If your server did that for every user, you'd be paying for it. Doing it in the browser means each student's own laptop does their own work, and your costs stay at roughly zero. This is the single most important architectural decision in the project.2. **Nothing is stored anywhere.** The audio goes to Groq, the text comes back, the document is built in memory and handed to the browser. If your server never stores a lecture recording, you can never leak one.3. **The pieces are sent in parallel, not one after another.** Four at a time. A 45-minute lecture would take four times longer if you did them in sequence.4. **The cuts land on silence, not on a stopwatch.** If you cut blindly at exactly 13:00 you might slice a word in half, and both pieces get that word wrong. Cutting in a gap between sentences costs nothing.5. **There's a middleman between the browser and Groq.** That middleman — the Netlify Function — exists for exactly one reason: to hold your API key somewhere the student can't see it. Part 14 covers what happens if you skip it.

---## Part 2 — Every tool, and why it's thereThis is the inventory. For each tool: what it is, what job it does in *your* project, and what you'd reach for instead if it vanished tomorrow.### The ones doing the actual work**ffmpeg** — the universal audio and video tool. It's a command-line program that can convert, cut, inspect, filter and analyse basically any media file that exists. You already know this one: it's the engine inside your `video_to_livephoto_batch.sh` script. In this project it does three jobs — measure the audio, find the silent gaps, and cut the pieces. *If it vanished:* nothing else comes close. This is the tool.**ffmpeg.wasm** — ffmpeg, recompiled so it runs inside a web browser. Normally ffmpeg is a program you install on a computer. WebAssembly ("wasm") is a way of taking a program written in C or C++ and packaging it so a browser can run it at close to native speed. That's how your site runs ffmpeg on a student's phone without them installing anything. *If it vanished:* you'd have to upload the whole 40 MB file to a server and split it there — slower for the student, and it would cost you money.**Groq Whisper** — the transcription service. Whisper is OpenAI's speech-to-text model; Groq runs it on their own very fast hardware and exposes it over an API. You send audio, you get text. *Alternatives:* OpenAI's own Whisper API, Deepgram, AssemblyAI, or running Whisper locally on your own machine (free, but far slower and your users can't use your laptop).**docx (the JavaScript library)** — builds Word documents in the browser. A `.docx` file is not simple text; it's actually a zip archive full of XML. This library hides that. *The Python equivalent, which you'll use in this notebook:* `python-docx`.### The ones holding it together**Netlify** — hosts your site and runs your serverless functions. "Static hosting" means it serves your HTML, CSS and JS files to anyone who visits. "Serverless functions" means small pieces of backend code that only run when someone calls them. *Alternatives:* Vercel, Cloudflare Pages, GitHub Pages (static only — no functions, so this project wouldn't work there).**Netlify Functions** — your backend. Three of them: `transcribe.mjs` (the middleman to Groq), `log-usage.mjs` (records that someone used the tool), `usage-list.mjs` (reads that log back for your admin page). They're "serverless" because you never manage a server — Netlify starts one when a request arrives and throws it away after.**Netlify Blobs** — a simple key-value store. Used only for the usage log. Think of it as a shared folder in the cloud your functions can read and write.**Git and GitHub** — version control and code hosting. Git records the history of your code; GitHub stores it online and is what Netlify watches. When you push, Netlify redeploys. That's notebook 3.### The plain web technologies**HTML** — the structure of the page. Buttons, headings, the file picker. Nouns.**CSS** — how it looks. Your whole retro lilac console theme is 608 lines of CSS. Adjectives.**JavaScript** — what happens when you interact with it. Click, upload, progress bar, download. Verbs.**Web Workers, COOP and COEP** — the awkward browser rules that let ffmpeg.wasm run at all. These caused the bug that broke your site on launch day. Part 13.

---## Part 3 — Setting up your workbenchRun the next cell. It checks what you've got and tells you exactly what to install if anything's missing. Nothing is installed automatically — you should always know what's going onto your machine.

In [ ]:
import shutil, subprocess, sysfrom pathlib import Pathprint("Python:", sys.version.split()[0])print("Running from:", Path.cwd())print()# --- command-line tools ---for tool in ["ffmpeg", "ffprobe", "say"]:    where = shutil.which(tool)    print(f"  {'OK  ' if where else 'MISSING'}  {tool:8} {where or ''}")print()# --- python packages ---for pkg, pipname in [("requests", "requests"), ("docx", "python-docx")]:    try:        __import__(pkg)        print(f"  OK       {pipname}")    except ImportError:        print(f"  MISSING  {pipname}   ->  pip install {pipname}")

**What each of those is for:**- `ffmpeg` — cuts and converts the audio. You already have this (your Live Photo script uses it). If it's missing: `brew install ffmpeg`.- `ffprobe` — comes with ffmpeg. It *inspects* audio instead of changing it. This is how you find out how long a file is, what format it's in, and so on.- `say` — a macOS built-in that turns text into speech. We'll use it to make a realistic test recording so you don't have to go and record a lecture right now.- `requests` — makes HTTP requests from Python. This is how you'll talk to Groq.- `python-docx` — builds Word documents. The Python twin of the `docx` library your site uses.If anything says MISSING, install it and re-run the cell before continuing.

In [ ]:
# Run this only if the cell above said something was missing.# The "!" prefix runs a shell command from inside Jupyter.# !pip install requests python-docx# !brew install ffmpeg

---### Pointing the notebook at your real codeThis notebook lives inside your project, at `academiccomback/learn/`. That means it can open and display your *actual* source files. Everything I show you from here on is read live off your disk — if you edit `transcribe.js`, these cells show the edit. Nothing here is a copy that can go stale.

In [ ]:
from pathlib import PathREPO = Path.cwd().parent          # learn/ sits inside the repo, so the repo is one level upWORK = Path.cwd() / "workspace"   # scratch space for the audio files we makeWORK.mkdir(exist_ok=True)print("Project root :", REPO)print("Scratch space:", WORK)print()expected = ["js/transcribe.js", "netlify/functions/transcribe.mjs", "transcribe.html", "netlify.toml"]for rel in expected:    print(f"  {'found  ' if (REPO/rel).exists() else 'MISSING'}  {rel}")

In [ ]:
def show(path, start_marker=None, end_marker=None, max_lines=60, note=None):    '''Print a slice of one of YOUR real source files, with line numbers.    We search for a piece of text rather than hard-coding line numbers,    so this keeps working after you edit the file.    '''    p = REPO / path    if not p.exists():        print(f"!! {p} not found"); return    lines = p.read_text(encoding="utf-8").splitlines()    start, end = 0, len(lines)    if start_marker:        for i, line in enumerate(lines):            if start_marker in line:                start = i                break        else:            print(f"!! couldn't find {start_marker!r} in {path}"); return    if end_marker:        for i in range(start + 1, len(lines)):            if end_marker in lines[i]:                end = i + 1                break    end = min(end, start + max_lines)    header = f"  {path}  (lines {start+1}-{end})  "    print("=" * len(header)); print(header); print("=" * len(header))    if note:        print(note + "\n")    for n in range(start, end):        print(f"{n+1:4} | {lines[n]}")    print()# Try it — this is the real top of your transcription script:show("js/transcribe.js", start_marker="/* Lecture Transcription", max_lines=6)

---## Part 4 — Making something to work withYou need audio. Two options:1. **Use a real lecture recording** — best, because it's the actual thing your tool has to handle. Drop the file into the `learn/workspace/` folder and set `SOURCE` to its name.2. **Generate speech with macOS `say`** — instant, and good enough to prove the pipeline works.The cell below does option 2. Change the text to whatever you like — try some of your Med Lab terminology and see how the transcription copes with it later. That's a genuinely useful experiment.

In [ ]:
import subprocessSCRIPT = (    "Good morning class. Today we are looking at the structure of the nephron "    "and how filtration actually happens in the kidney. The functional unit of "    "the kidney is the nephron, and each kidney contains roughly one million of them. "    "Blood arrives through the afferent arteriole, enters the glomerulus, and is "    "filtered under pressure into Bowman's capsule. Write that down, it comes up often.")raw = WORK / "sample.aiff"subprocess.run(["say", "-o", str(raw), SCRIPT], check=True)print("Made:", raw, f"({raw.stat().st_size/1024:.0f} KB)")# Convert to m4a so it looks like what a phone voice-memo app actually producesSOURCE = WORK / "lecture.m4a"subprocess.run(    ["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-c:a", "aac", "-b:a", "128k", str(SOURCE)],    check=True,)print("Converted:", SOURCE, f"({SOURCE.stat().st_size/1024:.0f} KB)")

**What just happened, line by line.**`subprocess.run([...])` is Python asking the operating system to run a command, exactly as if you'd typed it into Terminal. The list is the command split into pieces: `["say", "-o", "sample.aiff", "Good morning..."]` is the same as typing `say -o sample.aiff "Good morning..."`.Splitting it into a list instead of writing one long string matters more than it looks. If a filename contains a space, a single string would break in confusing ways; a list never does, because each item stays one item. Get into this habit now.`check=True` means "if the command fails, raise an error instead of carrying on silently". Without it, a failed command is invisible and you debug the wrong thing three cells later.The ffmpeg flags:- `-y` — overwrite the output file if it already exists, don't stop and ask- `-loglevel error` — only speak up if something goes wrong (ffmpeg is extremely chatty by default)- `-i <file>` — the **i**nput file- `-c:a aac` — **c**odec for **a**udio: use AAC, which is what iPhone voice memos use- `-b:a 128k` — **b**itrate for **a**udio: 128 kilobits per second- the last argument with no flag in front of it is always the outputThat last point is a real ffmpeg rule worth memorising: **input is flagged with `-i`, output is just the last thing on the line.**

---## Part 5 — What an audio file actually isBefore you can split audio sensibly you need to know what you're holding. Sound is a wave. A digital recording measures the height of that wave thousands of times per second and stores the numbers.- **Sample rate** — how many measurements per second. CD quality is 44,100 (44.1 kHz). Human speech carries almost all its information below 8 kHz, and there's a mathematical rule (Nyquist) that you need to sample at twice the highest frequency you care about. **16,000 Hz is therefore plenty for speech** — that's why your code uses it, and why Whisper itself works at 16 kHz internally. Sending 44.1 kHz audio would be nearly three times the data for zero extra accuracy.- **Channels** — 1 is mono, 2 is stereo. A lecturer is one person in one place. Stereo doubles your file size to record the same voice twice.- **Bitrate** — how much data per second of audio. Lower = smaller file, worse quality. 32 kbps is poor for music and completely fine for one person talking.- **Duration** — how long it is.`ffprobe` tells you all of these.

In [ ]:
import json, subprocessdef probe(path):    '''Ask ffprobe to describe an audio file, and hand back a Python dictionary.'''    out = subprocess.run(        ["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", str(path)],        capture_output=True, text=True, check=True,    )    return json.loads(out.stdout)info = probe(SOURCE)stream = info["streams"][0]print(f"format      : {info['format']['format_name']}")print(f"duration    : {float(info['format']['duration']):.2f} seconds")print(f"size        : {int(info['format']['size'])/1024:.0f} KB")print(f"codec       : {stream['codec_name']}")print(f"sample rate : {stream['sample_rate']} Hz")print(f"channels    : {stream['channels']}")print(f"bitrate     : {int(stream['bit_rate'])/1000:.0f} kbps")

**`capture_output=True, text=True`** tells Python to grab whatever the command printed and hand it back as a string instead of letting it spill onto the screen. `-of json` asks ffprobe to print its answer as JSON, which `json.loads` turns into a Python dictionary you can index into. This pattern — run a tool, capture its output, parse it — is one you'll use constantly.> **Try it yourself.** Run `probe()` on the original `sample.aiff` and compare. Which numbers changed when you converted to m4a, and why?

---## Part 6 — Why split the audio at all?A beginner's instinct is to send the whole lecture in one go. Three separate walls stop you, and each one alone would be enough.**1. Groq won't take it.** On the free tier the limit is 25 MB per file. A 45-minute lecture from a phone is comfortably over that.**2. Your serverless function gets killed at 10 seconds.** Netlify's free tier stops a function after 10 seconds. That's not 10 seconds of *your* code — it includes waiting for Groq to finish. Transcribing 45 minutes takes far longer. So each request must be small enough that Groq answers fast.**3. Speed.** Four pieces transcribed at the same time finish roughly four times sooner than four pieces done in a queue. Your code runs 4 at once — that's the `POOL = 4` setting.So: how big should a piece be? Your code targets ~13 minutes, and accepts anywhere from 6 to 14.5. Here's the reasoning, which is worth following because this is exactly the kind of trade-off you'll have to make on your own projects:- 13 minutes of mono 16 kHz audio at 32 kbps ≈ **3 MB**. Way under the 25 MB cap, with room for a chunk that runs long.- Fewer, bigger pieces means fewer requests, which means less chance of one failing and less total overhead.- But bigger pieces take Groq longer to process, pushing you towards that 10-second function timeout.- 13 minutes sits in the sweet spot.Let's confirm the size maths rather than trusting it.

In [ ]:
def estimate_chunk_size(seconds, kbps=32):    '''kilobits per second -> megabytes for a given duration.'''    kilobits = kbps * seconds    return kilobits / 8 / 1024      # -> kilobytes -> megabytesfor mins in [5, 13, 14.5, 30, 60]:    mb = estimate_chunk_size(mins * 60)    verdict = "OK" if mb < 25 else "TOO BIG for free tier"    print(f"{mins:>5} min  ->  {mb:5.1f} MB   {verdict}")

Notice that even a full hour would fit under 25 MB once it's squashed to mono 16 kHz 32 kbps. So the file-size cap is *not* actually the binding constraint — **the 10-second function timeout is**. That's a genuinely useful thing to understand: when several limits apply, only one of them is usually the one actually stopping you, and it's often not the obvious one.

---## Part 7 — Finding the natural pausesNow the interesting part. You want to cut at a moment when nobody is speaking.ffmpeg has a filter called `silencedetect`. It watches the audio and reports every stretch that stays quieter than a threshold for longer than a minimum duration. It doesn't change the audio — it just prints observations into the log.Two settings:- `noise=-30dB` — how quiet counts as "silence". Decibels here are negative, where 0 dB is the loudest possible and more negative is quieter. -30 dB is a sensible "nobody is talking" level. Set it too close to 0 and normal speech counts as silence; too negative and only a perfectly silent room qualifies.- `d=0.4` — the gap must last at least 0.4 seconds. Shorter than that and you'd match the tiny gaps between ordinary words.The trick in the command: `-f null -` means "produce no output file". We only want the log.

In [ ]:
import redef detect_silences(path, noise="-30dB", min_dur=0.4):    '''Run ffmpeg's silencedetect and pull the results out of its log.    Returns (duration_in_seconds, [list of silent stretches]).    '''    result = subprocess.run(        ["ffmpeg", "-i", str(path), "-af", f"silencedetect=noise={noise}:d={min_dur}", "-f", "null", "-"],        capture_output=True, text=True,    )    log = result.stderr        # ffmpeg writes its log to stderr, not stdout    duration = 0.0    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", log)    if m:        duration = int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))    silences, pending = [], None    for line in log.splitlines():        s = re.search(r"silence_start:\s*(-?\d+(?:\.\d+)?)", line)        if s:            pending = float(s.group(1))        e = re.search(r"silence_end:\s*(-?\d+(?:\.\d+)?)", line)        if e and pending is not None:            end = float(e.group(1))            silences.append({"start": pending, "end": end, "mid": (pending + end) / 2})            pending = None    return duration, silencesduration, silences = detect_silences(SOURCE)print(f"Duration: {duration:.2f}s")print(f"Found {len(silences)} silent stretches:\n")for s in silences:    print(f"  {s['start']:7.2f}s -> {s['end']:7.2f}s   (gap of {s['end']-s['start']:.2f}s, midpoint {s['mid']:.2f}s)")

**Three things in that function worth pausing on.****ffmpeg writes its log to stderr, not stdout.** Every program has two output channels: stdout for results, stderr for messages. ffmpeg treats its entire running commentary as messages. If you read `result.stdout` here you get an empty string and a very confusing half hour. This trips up nearly everyone once.**Regular expressions.** `r"silence_start:\s*(-?\d+(?:\.\d+)?)"` is a pattern for finding text. Broken down: `silence_start:` matches those literal characters; `\s*` allows any amount of whitespace; the brackets `( )` mark the part we want to keep; `-?` an optional minus sign; `\d+` one or more digits; `(?:\.\d+)?` an optional dot-followed-by-digits. Together: "find `silence_start:` and capture the number after it, whether or not it has a decimal point or a minus sign."**Start and end arrive on separate lines.** ffmpeg prints `silence_start` when a gap begins and `silence_end` when it finishes — two different log lines. So the loop holds onto the start in `pending` until the matching end shows up, then pairs them. The `mid` is what you actually cut on: the middle of a gap is the safest possible moment to slice.> **Try it yourself.** Re-run with `noise="-50dB"` (much stricter) and `noise="-20dB"` (much looser). Watch the number of detected silences change. Then think about what a lecture hall with an air conditioner running would do to this.

---## Part 8 — Deciding where to cutYou now have a list of possible cut points. You need to choose.The rule your code follows, in plain English: *walk forward through the recording aiming to cut every 13 minutes; each time, look for a silence near that 13-minute mark and cut there instead; if there isn't one nearby, just cut at 13 minutes anyway.*"Nearby" means within 90 seconds either side, and never producing a piece shorter than 6 minutes or longer than 14.5.This next function is a **direct translation of `planCuts` in your real `transcribe.js`** — same logic, same variable names, same constants. Read it, then we'll look at the original.

In [ ]:
TARGET_SEC    = 780   # 13 minutes — where we'd ideally cutMIN_SEC       = 360   # 6 minutes — never make a piece shorter than thisMAX_SEC       = 870   # 14.5 minutes — never make a piece longer than thisSEARCH_WINDOW = 90    # how far either side of the ideal point we'll hunt for silencedef plan_cuts(duration, silences):    '''Work out the timestamps to cut at. Returns e.g. [0, 778.4, 1559.1, 2100.0].'''    # Short enough to send in one piece? Then there's nothing to plan.    if not duration or duration <= MAX_SEC:        return [0, duration or 1]    cuts = [0]    pos = 0    while duration - pos > MAX_SEC:        ideal = pos + TARGET_SEC        lo = max(pos + MIN_SEC, ideal - SEARCH_WINDOW)   # earliest acceptable cut        hi = min(pos + MAX_SEC, ideal + SEARCH_WINDOW)   # latest acceptable cut        # Of the silences inside that window, take the one closest to ideal.        best, best_dist = None, float("inf")        for s in silences:            if s["mid"] < lo or s["mid"] > hi:                continue            dist = abs(s["mid"] - ideal)            if dist < best_dist:                best_dist, best = dist, s["mid"]        cut = best if best is not None else ideal   # no silence nearby? cut anyway.        cuts.append(cut)        pos = cut    cuts.append(duration)    return cuts# Your sample is far too short to be split, so let's invent a realistic lecture# to watch the algorithm actually work.import randomrandom.seed(7)fake_duration = 45 * 60                       # a 45-minute lecturefake_silences = []t = 20while t < fake_duration:    t += random.uniform(15, 70)               # a pause every 15-70 seconds    gap = random.uniform(0.4, 2.0)    fake_silences.append({"start": t, "end": t + gap, "mid": t + gap / 2})    t += gapcuts = plan_cuts(fake_duration, fake_silences)print(f"A {fake_duration/60:.0f}-minute lecture with {len(fake_silences)} pauses in it")print(f"Cut points: {[round(c,1) for c in cuts]}\n")for i in range(len(cuts) - 1):    length = cuts[i+1] - cuts[i]    landed = "on a pause" if any(abs(s["mid"] - cuts[i+1]) < 0.01 for s in fake_silences) else "no pause nearby - forced"    print(f"  piece {i+1}: {cuts[i]/60:5.1f} min -> {cuts[i+1]/60:5.1f} min   ({length/60:4.1f} min, cut {landed})")

In [ ]:
# A quick picture of where those cuts landed.width = 100print("The whole lecture, left to right. '|' marks a cut.\n")bar = ["-"] * widthfor c in cuts[1:-1]:    bar[int(c / fake_duration * (width - 1))] = "|"print("".join(bar))print(f"0 min{' ' * (width - 14)}{fake_duration/60:.0f} min")

### Now the real thingHere is that exact algorithm as it exists in your site, read straight off your disk:

In [ ]:
show("js/transcribe.js",     start_marker="function planCuts",     end_marker="return cuts;",     note="The JavaScript original. Compare it line by line with the Python above.")

**The differences are almost entirely cosmetic**, and that's the point of this whole exercise. Line them up:| Python | JavaScript | Same idea? ||---|---|---|| `def plan_cuts(duration, silences):` | `function planCuts(duration, silences) {` | yes || `if not duration or duration <= MAX_SEC:` | `if (!duration \|\| duration <= MAX_SEC)` | yes — `not` is `!`, `or` is `\|\|` || `cuts = [0]` | `const cuts = [0];` | yes — JS wants you to say `const` or `let` || `while duration - pos > MAX_SEC:` | `while (duration - pos > MAX_SEC) {` | yes — JS puts the condition in brackets || `max(...)`, `min(...)`, `abs(...)` | `Math.max(...)`, `Math.min(...)`, `Math.abs(...)` | yes — JS keeps its maths in a `Math` box || `float("inf")` | `Infinity` | yes || indentation defines the block | `{ }` defines the block | yes || `None` | `null` | yes |Once you've seen that table, JavaScript stops being a different language and becomes Python wearing curly brackets. Almost everything else you'll meet is a variation on these.

---## Part 9 — Cutting the piecesYou know where to cut. Now do it.Each piece gets extracted and re-encoded in a single ffmpeg command. Three separate things happen:**Extract the time range.** `-ss <start> -to <end>`.There's a subtlety here worth knowing because it bites people. Putting `-ss` **before** `-i` means "skip to this point *before* you start decoding" — fast, because ffmpeg jumps straight there. Putting it **after** `-i` means "decode from the beginning and throw away everything before this point" — accurate but slow on a long file. Your code puts it before, which is the right call.**Clean up the sound.** `-af "highpass=f=80,dynaudnorm=f=200:g=15"`. The `-af` is "audio filter", and the comma chains filters one into the next:- `highpass=f=80` throws away everything below 80 Hz. Human speech essentially doesn't live down there, but air conditioning, traffic rumble and the thump of a phone being nudged all do. Free improvement.- `dynaudnorm` evens out the volume over time. When the lecturer walks away from the phone and goes quiet, this brings them back up. This matters enormously for real lecture recordings.**Shrink it.** `-ac 1` (mono), `-ar 16000` (16 kHz), `-b:a 32k` (32 kbps) — the settings from Part 5.

In [ ]:
AUDIO_FILTERS = "highpass=f=80,dynaudnorm=f=200:g=15"def cut_chunk(src, start, end, out_path, filters=AUDIO_FILTERS):    '''Extract start->end from src, clean it up, and write a small mono mp3.'''    cmd = [        "ffmpeg", "-y", "-loglevel", "error",        "-ss", f"{start:.3f}",      # BEFORE -i  = fast seek        "-to", f"{end:.3f}",        "-i", str(src),    ]    if filters:        cmd += ["-af", filters]    cmd += ["-ac", "1", "-ar", "16000", "-b:a", "32k", str(out_path)]    print("  $ " + " ".join(cmd))          # see the actual command being run    subprocess.run(cmd, check=True)    return out_path# Cut our short sample into one piece (it's too short to need splitting).real_duration, real_silences = detect_silences(SOURCE)real_cuts = plan_cuts(real_duration, real_silences)print(f"Duration {real_duration:.2f}s -> cuts at {[round(c,2) for c in real_cuts]}\n")chunks = []for i in range(len(real_cuts) - 1):    out = WORK / f"chunk_{i:02d}.mp3"    cut_chunk(SOURCE, real_cuts[i], real_cuts[i+1], out)    chunks.append(out)    print(f"  -> {out.name}  ({out.stat().st_size/1024:.0f} KB)\n")

Notice the cell prints the exact command before running it. Do this whenever you're driving a command-line tool from code — when something breaks, you can copy that line straight into Terminal and poke at it directly. It turns a mysterious Python error into a normal ffmpeg problem you can actually debug.> **Try it yourself.** Run `cut_chunk` twice on the same range, once with `filters=AUDIO_FILTERS` and once with `filters=None`. Compare file sizes, then listen to both. Can you hear what the highpass removed?

In [ ]:
# Here's the same operation in your real code:show("js/transcribe.js", start_marker="const outName = `out_", end_marker="chunks.push", max_lines=22,     note="Same flags, same order, same reasoning. Just JavaScript syntax.")

---## Part 10 — Sending it to GroqNow the actual transcription.### First: your API keyAn API key is a password that proves a request is coming from your account. Groq bills against it. Three rules, and the first one is the one people break:1. **Never put it in your code.** Not in a Python file, not in a notebook cell, not in JavaScript. If it ends up in git it is permanently in your repository's history, even if you delete it in a later commit — and your repo is public.2. **Keep it in an environment variable or type it in when needed.**3. **If you ever leak one, revoke it immediately** in the Groq console and make a new one. Don't hope nobody noticed; bots scan public GitHub for exactly this, within minutes.The cell below uses `getpass`, which pops up a hidden input box. What you type is never displayed, never saved into the notebook file, and disappears when you shut the kernel down.

In [ ]:
import os, getpassGROQ_API_KEY = os.environ.get("GROQ_API_KEY")if not GROQ_API_KEY:    GROQ_API_KEY = getpass.getpass("Paste your Groq API key (it won't be shown): ").strip()print("Key loaded." if GROQ_API_KEY else "No key — the transcription cells below will be skipped.")print(f"(starts with {GROQ_API_KEY[:4]}..., {len(GROQ_API_KEY)} characters)" if GROQ_API_KEY else "")

Only the first four characters get printed, as a sanity check that you pasted the right thing without putting the key on screen. Small habit, worth keeping.If you'd rather not retype it every session, put it in your shell profile instead — then `os.environ.get` picks it up automatically:```bashecho 'export GROQ_API_KEY="your-key-here"' >> ~/.zshrcsource ~/.zshrc```### The requestGroq's transcription endpoint follows the same shape as OpenAI's, so what you learn here transfers. You send a **multipart form** — the format a web form uses when it has a file attached — containing the audio plus some settings.Every parameter, and what it does:| Parameter | What it does ||---|---|| `file` | the audio itself || `model` | which Whisper to use. `whisper-large-v3-turbo` is fast and cheap; `whisper-large-v3` is slower, pricier, slightly more accurate || `language` | the language, as a two-letter code. **Supplying this is the single biggest accuracy win available to you.** Left out, Whisper spends effort guessing, and guesses badly on accented English || `temperature` | how much the model is allowed to improvise. `0` means "give me your most likely answer, don't get creative". Always 0 for transcription || `response_format` | `json` gives you `{"text": "..."}`. `verbose_json` adds timestamps per segment || `prompt` | up to 224 tokens of context to steer spelling and vocabulary. This is the lever for medical terminology — see the experiment below |

In [ ]:
import requestsGROQ_URL = "https://api.groq.com/openai/v1/audio/transcriptions"def transcribe(path, api_key, language="en", temperature=0, prompt=None,               model="whisper-large-v3-turbo"):    '''Send one audio file to Groq and return the transcribed text.'''    data = {        "model": model,        "response_format": "json",        "temperature": str(temperature),    }    if language:        data["language"] = language    if prompt:        data["prompt"] = prompt    with open(path, "rb") as fh:                       # "rb" = read, binary        response = requests.post(            GROQ_URL,            headers={"Authorization": f"Bearer {api_key}"},            files={"file": (path.name, fh, "audio/mpeg")},            data=data,            timeout=120,        )    if response.status_code != 200:        raise RuntimeError(f"Groq said {response.status_code}: {response.text[:400]}")    return response.json()["text"]if GROQ_API_KEY:    text = transcribe(chunks[0], GROQ_API_KEY)    print(text)else:    print("(skipped — no API key)")

**Reading that request piece by piece:**- `headers={"Authorization": f"Bearer {api_key}"}` — headers are extra information attached to a request. `Bearer <token>` is the standard way of saying "here's my credential". The word "Bearer" is literal and required.- `files={...}` — this is what makes it a multipart upload. The tuple is `(filename, file_object, content_type)`.- `with open(path, "rb") as fh:` — `with` guarantees the file gets closed even if something fails inside the block. Always open files this way.- `timeout=120` — give up after two minutes rather than hanging forever. **Always set a timeout on a network call.** Without one, a single unresponsive server can freeze your program indefinitely.- `response.status_code` — 200 means success. 4xx means you did something wrong (401 = bad key, 429 = too many requests). 5xx means their end broke.

### Experiment: does the `prompt` parameter fix medical vocabulary?This is the experiment that matters most for your actual users. Whisper has heard far more general English than it has heard histopathology lectures, so it tends to "correct" technical terms into everyday words that sound similar.Let's test whether feeding it the vocabulary in advance helps. Run the cell — it transcribes the same audio twice, once cold and once primed.

In [ ]:
if GROQ_API_KEY:    MEDICAL = (        "Good morning. Today's histopathology lecture covers the nephron, glomerulus, "        "Bowman's capsule, afferent and efferent arterioles, the loop of Henle, "        "eosinophilic cytoplasm, anisocytosis and poikilocytosis."    )    med_raw = WORK / "medical.aiff"    med_mp3 = WORK / "medical.mp3"    subprocess.run(["say", "-o", str(med_raw), MEDICAL], check=True)    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(med_raw),                    "-ac", "1", "-ar", "16000", "-b:a", "32k", str(med_mp3)], check=True)    print("WHAT WAS ACTUALLY SAID:")    print(" ", MEDICAL, "\n")    print("WITHOUT a prompt:")    print(" ", transcribe(med_mp3, GROQ_API_KEY), "\n")    hint = "Histopathology lecture. Terms: nephron, glomerulus, Bowman's capsule, afferent arteriole, efferent arteriole, loop of Henle, eosinophilic, anisocytosis, poikilocytosis."    print("WITH a vocabulary prompt:")    print(" ", transcribe(med_mp3, GROQ_API_KEY, prompt=hint))else:    print("(skipped — no API key)")

Compare the two carefully, word by word. If the primed version gets more of the terminology right, you've just proved the case for adding a "what's this lecture about?" box to your site — and you proved it with an experiment rather than an opinion. That's worth more than any advice I could give you.One caveat you should know about: on a chunk that's mostly silence, Whisper sometimes outputs the prompt text itself as if it had heard it. So if you do build this feature, test it against a recording with long quiet stretches before you ship it.

---## Part 11 — Stitching, and building the documentTwo small steps left.**Stitching** is just joining the pieces back together in the right order. The only thing that matters is *order* — piece 3 must not end up before piece 2. Your JavaScript handles this by writing each result into a fixed slot in an array rather than appending as they finish, because the pieces come back in whatever order Groq happens to answer.That's a real bug waiting to happen and worth internalising: **when you run things in parallel, they finish out of order.** If you append results as they arrive, your transcript is shuffled. Pre-allocate the slots.

In [ ]:
# How your JS keeps the order straight, in Python terms:results = [None] * 4                 # four empty slots, one per piece# Imagine these come back in a jumbled order — piece 3 finished first:results[2] = "third piece text"results[0] = "first piece text"results[3] = "fourth piece text"results[1] = "second piece text"transcript = "\n\n".join(t.strip() for t in results if t)print(transcript)

In [ ]:
show("js/transcribe.js", start_marker="const results = new Array", end_marker="showDone(allChunks.length);",     max_lines=35, note="The real parallel-with-order code. 'results[i] = ...' is the important line.")

### Building the .docxA `.docx` is a zip file containing XML. You never want to write that by hand. `python-docx` does it for you; your site uses the JavaScript `docx` library, which works almost identically.Your document has four parts, in this order: a title, a disclaimer explaining that this is a raw transcript, a ready-made prompt the student can paste into an AI to tidy it up, and then the transcript itself.That third part is the cleverest thing in your product, by the way. You don't attempt to clean the transcript yourself — you hand the student a prompt that makes someone else's AI do it. Costs you nothing, and it's more useful than a mediocre auto-summary would be.

In [ ]:
from docx import Documentfrom docx.shared import Pt, RGBColordef build_docx(transcript_text, out_path):    doc = Document()    doc.add_heading("Lecture Transcript", level=1)    p = doc.add_paragraph()    run = p.add_run("Generated free by AcademicComback — a Mr Cee project.")    run.italic = True    run.font.color.rgb = RGBColor(0x7A, 0x5F, 0xC0)      # your lilac    doc.add_heading("Read this first", level=2)    doc.add_paragraph(        "This tool only transcribes — it doesn't clean anything up. What follows is a "        "raw, word-for-word transcript. To turn it into proper notes, copy the prompt "        "below into any AI, paste your transcript where it says to, and it'll organise it."    )    doc.add_heading("Prompt — paste this into your AI", level=2)    prompt_p = doc.add_paragraph()    prompt_run = prompt_p.add_run(        "Turn the transcript below into clear, comprehensive lecture notes a student "        "could study from. Extract the real teaching content, cut the filler, reorganise "        "by topic rather than by the order things were said, and finish with a short "        "Key Takeaways section.\n\nTranscript:\n[paste your transcript here]"    )    prompt_run.font.name = "Courier New"    prompt_run.font.size = Pt(10)    doc.add_heading("Raw transcript", level=2)    for para in transcript_text.split("\n\n"):        if para.strip():            doc.add_paragraph(para.strip())    doc.save(out_path)    return out_pathsample_text = text if (GROQ_API_KEY and "text" in dir()) else "This is placeholder transcript text."out = build_docx(sample_text, WORK / "my-transcript.docx")print("Wrote:", out, f"({out.stat().st_size/1024:.1f} KB)")print("\nOpen it:")print(f"  !open '{out}'")

In [ ]:
# Proof that a .docx really is just a zip full of XML — have a look inside yours.import zipfilewith zipfile.ZipFile(WORK / "my-transcript.docx") as z:    for name in z.namelist():        print(f"  {name}")

That's the whole mystery of Word documents. `word/document.xml` is your actual text; everything else is styling and bookkeeping. Now you know why you use a library.

---## Part 12 — You just rebuilt it. Here's the original.Stop and take stock. You have now written, and run, every meaningful step of your own product:| Step | Your Python | Your site's JavaScript ||---|---|---|| Inspect the audio | `probe()` | inside `analyze()` || Find the pauses | `detect_silences()` | `analyze()` || Choose cut points | `plan_cuts()` | `planCuts()` || Cut the pieces | `cut_chunk()` | inside `chunkFile()` || Transcribe | `transcribe()` | `transcribeChunk()` + `transcribe.mjs` || Stitch | `"\n\n".join(...)` | `results.map(...).join("\n\n")` || Build the document | `build_docx()` | `buildDocx()` |Now read the whole of the real `run()` function. This is the conductor — the function that calls everything else in order. You should recognise every single line of it.

In [ ]:
show("js/transcribe.js", start_marker="async function run()", end_marker="showDone(allChunks.length);",     max_lines=75, note="The conductor. Every step you built, in sequence.")

**Two JavaScript-specific things in there worth explaining properly, because they'll confuse you otherwise.****`async` and `await`.** Some operations take time — reading a file, calling an API. In JavaScript these don't stop the world; the page has to stay responsive while they happen. `await` means "pause *this function* here until that's finished, but let the rest of the page carry on." A function containing `await` must be marked `async`.Python has exactly the same keywords with exactly the same meaning. You just didn't need them here, because a notebook is happy to sit and block.**`try` / `catch`.** Identical to Python's `try` / `except`, different spelling. `try { ... } catch (err) { ... }` means "attempt this; if it fails, run that instead of crashing."Look at how `run()` uses it: each risky stage is wrapped separately, so the user gets a message about *what* failed rather than one generic error. Notice this though —```js} catch (err) {  return fail("Couldn't load the audio engine. Check your connection and try again.");}```— the real error in `err` is thrown away. That's why your launch-day bug was so hard to track down: the page said "check your connection" when the actual problem was an HTTP header. A useful habit for your own code: keep the friendly message for the user, but also `console.error(err)` so the real cause is there in the browser console when you need it.> **Try it yourself.** Open `js/transcribe.js` and add `console.error(err)` inside each catch block. Small change, and it would have saved hours on launch day.

---## Part 13 — The three things the browser makes hardEverything you did above was straightforward in Python. In a browser, three of those steps are genuinely awkward. This section is the part you cannot learn from the Python version, and it's the part that broke your site on launch day.### 1. ffmpeg has to be smuggled inYou typed `ffmpeg` in the notebook because it's installed on your Mac. A student's browser has no ffmpeg and can't install one.The answer is WebAssembly. ffmpeg is written in C. WebAssembly is a compilation target that browsers can execute at near-native speed. So `ffmpeg.wasm` is genuinely ffmpeg — the same C code — compiled into a form a browser can run. It arrives as a ~30 MB download the first time, which is why your page says "Loading the audio engine".### 2. It needs its own thread, and browsers are paranoid about thatSplitting audio is heavy. If it ran on the same thread as the page, the whole interface would freeze — no progress bar, no scrolling, nothing.So ffmpeg.wasm runs in a **Web Worker**: a separate thread, running in the background, that talks to the page by passing messages back and forth. That's what keeps your progress bar moving.Workers need to share memory with the page efficiently, using something called `SharedArrayBuffer`. And here's where it gets political: `SharedArrayBuffer` was switched off in every browser after the Spectre security vulnerability in 2018, because it could be abused to read memory a page shouldn't see. You can have it back, but only if you prove your page is properly isolated from other sites. You prove that with two HTTP headers:- `Cross-Origin-Opener-Policy: same-origin`- `Cross-Origin-Embedder-Policy: require-corp`Those two lines in your `netlify.toml` are the entire reason your transcription page works.

In [ ]:
show("netlify.toml", note="Every line here exists because of a specific failure. See below.")

### 3. The bug that broke launch day — worth understanding properlyWhen your site first went live, the transcription page said *"Couldn't load the audio engine."* There were actually **three separate faults stacked on top of each other**, and each only became visible after the one in front of it was fixed. This is very typical of real debugging, so it's worth walking through.**Fault one — the headers were on the wrong address.** The rule said `for = "/transcribe.html"`. But Netlify serves that page at the pretty URL `/transcribe`, with no `.html`. Those are different strings, so the rule never matched, so the page never got its headers, so `SharedArrayBuffer` stayed switched off. Fix: `for = "/transcribe*"`, which matches both.*The lesson:* a rule that matches on an exact path is a rule that silently does nothing the moment the path changes.**Fault two — the worker was coming from someone else's website.** With the headers finally working, a new failure appeared. The ffmpeg library was being loaded from a CDN (`unpkg.com`), and it starts its own worker by fetching a second file from that same CDN. But `require-corp` means "don't load anything from other sites unless they explicitly allow it" — so the browser blocked it. Fix: download those files into the project (`js/vendor/ffmpeg/`) and serve them yourself, so they're no longer "someone else's website".*The lesson:* loading libraries from a CDN is convenient right up until a security policy disagrees. Self-hosting removes a whole category of problem.**Fault three — and this one is genuinely obscure.** Even served from your own site, the worker was *still* blocked, with `net::ERR_BLOCKED_BY_RESPONSE`.The reason: when a page has `Cross-Origin-Embedder-Policy` switched on, a worker script must carry that header **on its own response** too. Not just the page — the worker file itself. A worker gets its own little world, and that world has to opt in to the same policy independently.The proof it was that, and not something wrong with the ffmpeg file specifically: `js/namegate.js` — an ordinary script that works fine everywhere else on your site — failed in exactly the same way when loaded as a worker. That ruled out the file and pointed at the policy. **When two unrelated things fail identically, the cause is something they share.** That's the most useful debugging instinct in this entire notebook.Fix: the second headers block in `netlify.toml`, adding `Cross-Origin-Embedder-Policy = "require-corp"` to the vendored ffmpeg files.> **Worth sitting with:** each fix revealed the next fault. You can't see bug two until bug one is gone. This is normal, it is not a sign you're doing it wrong, and the only way through is one layer at a time.

---## Part 14 — Why the API key can't live in the browserThis is the most important security idea in the project, and it generalises to everything you'll ever build.**Everything in the browser is public.** All of it. Your HTML, your CSS, your JavaScript — the browser downloads it all onto the user's machine to run it. Anyone can press F12 and read every line. There is no such thing as a secret in front-end code.So if you wrote this:```javascript// NEVER do thisconst GROQ_API_KEY = "gsk_abc123...";fetch("https://api.groq.com/...", { headers: { Authorization: `Bearer ${GROQ_API_KEY}` }});```...your key is visible to every visitor. Someone finds it, uses your quota, and you find out when the bill or the rate-limit arrives.**The fix is a middleman that the user can't see inside.** Your Netlify Function runs on Netlify's servers, not in the browser. The key lives there as an environment variable. The browser sends audio to *your* function; your function attaches the key and forwards it to Groq.The browser never sees the key. It only ever sees an address on your own site.

In [ ]:
show("netlify/functions/transcribe.mjs",     note="Your whole backend, top to bottom. Every line is doing one of: check, guard, forward, translate errors.")

**Walk through what it does, in order:**1. **Reject anything that isn't a POST.** GET is for fetching, POST is for sending. This endpoint only accepts sends.2. **Check the key exists.** `process.env.GROQ_API_KEY` reads the environment variable you set in the Netlify dashboard. If it's missing, say so clearly — a confusing 500 error here would waste an afternoon.3. **Read the audio** out of the request body.4. **Refuse empty payloads.** Cheap guard, saves a pointless call to Groq.5. **Build the form** — same fields you used in Python: file, model, language, temperature.6. **Forward it to Groq** with the key attached.7. **Translate the errors.** A 429 from Groq becomes a friendly "quota is briefly maxed out". This matters: the raw upstream error is meaningless to a student.8. **Return just the text.**Note what it deliberately *doesn't* do: it never writes the audio to disk, never logs the transcript, never keeps anything. Audio in, text out, nothing retained. That's what lets you promise "nothing is stored" honestly.And note the design constraint in the comment at the top — the function must do nothing but forward, because of that 10-second timeout. Any extra work here and long chunks would start failing.> **Try it yourself.** Open your live site, press F12, go to the Network tab, and run a transcription. Watch the request to `/api/transcribe`. You can see the audio going out and the text coming back — and you cannot see your Groq key anywhere, because it never leaves Netlify's servers.

---## Part 15 — Rebuild it yourselfReading and running someone else's steps is the easy part. These exercises are the actual learning. Do them without AI. Getting stuck for twenty minutes and then working it out is the entire point — that struggle is what makes it stick.### Level 1 — command line only, no PythonOpen Terminal and do the whole pipeline by hand. Every one of these maps to something the notebook did for you.```bash# 1. how long is it, and what is it?ffprobe -v error -show_format -show_streams -of json lecture.m4a# 2. where are the pauses?ffmpeg -i lecture.m4a -af silencedetect=noise=-30dB:d=0.4 -f null -# 3. cut from 2:00 to 4:00, cleaned up and shrunkffmpeg -ss 120 -to 240 -i lecture.m4a \  -af "highpass=f=80,dynaudnorm=f=200:g=15" \  -ac 1 -ar 16000 -b:a 32k piece1.mp3# 4. transcribe itcurl -s https://api.groq.com/openai/v1/audio/transcriptions \  -H "Authorization: Bearer $GROQ_API_KEY" \  -F "file=@piece1.mp3" \  -F "model=whisper-large-v3-turbo" \  -F "language=en" \  -F "temperature=0"```If you can run those four commands from memory and explain each flag, you understand this pipeline better than most people who'd call themselves web developers.### Level 2 — Python, from a blank fileClose this notebook. Open a new empty `.py` file and write a script that takes an audio filename and produces a `.docx`, without looking back here. When you get stuck, look up the *documentation* — not this notebook, and not an AI.Checkpoints, in order:1. It prints the duration of any audio file you give it2. It prints the silences3. It decides cut points and prints them4. It writes the chunk files to disk5. It transcribes one chunk6. It transcribes all chunks, in the right order7. It produces a .docx### Level 3 — change your actual siteSmall, real changes to `js/transcribe.js`. Each one is genuinely useful:1. Add `console.error(err)` to every `catch` block, so the real error is visible next time.2. Change `TARGET_SEC` from 780 to 600 and work out, before you test it, how many pieces a 45-minute lecture will produce.3. Add a word count to the progress text.4. **The big one:** add a text box asking "What's this lecture about?" and pass what they type through to Groq as the `prompt` parameter. You proved in Part 10 that this helps with medical vocabulary. You'd need to touch `transcribe.html` (the box), `js/transcribe.js` (read it, send it) and `netlify/functions/transcribe.mjs` (pass it on). That's a full-stack feature, end to end — and it's the single most valuable thing you could add for your actual users.### Level 4 — explain itExplain the whole pipeline out loud to someone who doesn't code, in under five minutes, with no notes. If you hit a step where you have to say "and then some magic happens", that's your next thing to study.

---## Part 16 — Reference cardKeep this. It's everything above, compressed.### ffmpeg flags you'll actually use| Flag | Meaning ||---|---|| `-i <file>` | input file || `-y` | overwrite output without asking || `-loglevel error` | only tell me about problems || `-ss <sec>` | start at (put it **before** `-i` for speed) || `-to <sec>` | stop at || `-t <sec>` | run for this long (instead of `-to`) || `-ac 1` | mono || `-ar 16000` | 16 kHz sample rate || `-b:a 32k` | audio bitrate || `-af "..."` | audio filter chain, comma-separated || `-f null -` | produce no file — I only want the log || `-c:a copy` | copy audio without re-encoding (fast, lossless) |### Useful audio filters| Filter | What it's for ||---|---|| `silencedetect=noise=-30dB:d=0.4` | report quiet stretches || `highpass=f=80` | remove rumble below 80 Hz || `lowpass=f=8000` | remove hiss above 8 kHz || `dynaudnorm=f=200:g=15` | even out changing volume || `loudnorm` | normalise to broadcast loudness || `atempo=1.5` | speed up without changing pitch |### Groq transcription API```POST https://api.groq.com/openai/v1/audio/transcriptionsAuthorization: Bearer <key>file          the audio (multipart)model         whisper-large-v3-turbo | whisper-large-v3language      entemperature   0prompt        up to 224 tokens of vocabulary hintsresponse_format   json | verbose_json | text```Limits: 25 MB per file free tier, 100 MB dev tier. Minimum billed length 10 seconds.### HTTP status codes worth memorising| Code | Meaning ||---|---|| 200 | fine || 400 | your request was malformed || 401 | bad or missing credential || 403 | authenticated, but not allowed || 404 | no such thing || 429 | slow down, too many requests || 500 | the server broke || 502 | a server upstream of it broke |### Python patterns from this notebook```pythonsubprocess.run([...], capture_output=True, text=True, check=True)   # run a CLI toolresult.stderr                        # ffmpeg logs here, NOT stdoutre.search(r"pattern (\d+)", text)    # pull a number out of textwith open(path, "rb") as fh:         # always use `with`os.environ.get("KEY")                # secrets from the environment, never in coderequests.post(url, headers=..., files=..., data=..., timeout=120)```### Where to look things up (not an AI)- ffmpeg filters — <https://ffmpeg.org/ffmpeg-filters.html>- Groq speech-to-text — <https://console.groq.com/docs/speech-to-text>- MDN, for anything web — <https://developer.mozilla.org>- python-docx — <https://python-docx.readthedocs.io>- Netlify Functions — <https://docs.netlify.com/functions/overview/>MDN especially. It is the reference for HTML, CSS and JavaScript, it's free, and it's better than most paid courses.---**Next:** notebook 2 covers Tabitha and the front end — HTML structure, the CSS theme, and how JavaScript actually manipulates a page. Notebook 3 covers git, GitHub and deploying.